In [1]:

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from data.get_data import get_dataframe
from data_processor.calculate_stats import calculate_statistics
from data_processor.data_categorising import categories_columns
from data_processor.data_cleaner import clean_data
from data_processor.fight_stats import finalProcessingForFighter, calculateAverages
from data_processor.data_types_fixes import check_and_process_data_type, drop_col_for_training

In [2]:
og_df = get_dataframe('original.csv')

In [3]:
cleaned_df = clean_data(og_df)

In [4]:
stats_df = calculate_statistics(cleaned_df)

In [5]:
processed_df = finalProcessingForFighter(stats_df)

In [6]:
processed_df = check_and_process_data_type(processed_df)

In [7]:
avg_df = calculateAverages(processed_df)

In [8]:
cat_df = categories_columns(avg_df)

In [9]:
df_for_training = drop_col_for_training(cat_df)

In [10]:
df_for_training = df_for_training.sort_index()
df_for_training

,Total_KD,Total_STR,Total_TD,Total_SUB,Opp_KD,Opp_STR,Opp_TD,Opp_SUB,Target,Avg_Round_Time,Avg_Round,Opp_Avg_Round_Time,Opp_Avg_Round,Weight_Class_code
0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0.0,0.0,8
1,0,0,0,0,0,0,0,0,0,0.000000,0.000000,170.0,1.0,8
2,0,0,0,0,0,0,0,0,0,0.000000,0.000000,747.0,1.0,8
3,0,0,0,0,0,0,0,0,0,0.000000,0.000000,591.0,1.0,8
4,0,0,0,0,0,0,0,0,0,0.000000,0.000000,454.5,1.5,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14819,1,64,0,0,0,0,0,0,0,309.666667,2.333333,0.0,0.0,5
14820,0,275,9,5,1,289,2,0,0,445.875000,3.750000,493.6,5.0,10
14821,0,76,0,0,0,226,6,2,0,300.000000,3.000000,486.5,5.5,13
14822,1,142,2,1,1,546,14,11,0,295.250000,3.250000,313.0,4.0,11


In [11]:
X = df_for_training.drop("Target", axis=1)
y = df_for_training["Target"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [25]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.6,
    colsample_bytree=0.8,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

[0]	validation_0-logloss:0.69255


[50]	validation_0-logloss:0.68113
[100]	validation_0-logloss:0.67563
[150]	validation_0-logloss:0.67275
[200]	validation_0-logloss:0.67152
[250]	validation_0-logloss:0.67086
[300]	validation_0-logloss:0.67038
[350]	validation_0-logloss:0.67008
[400]	validation_0-logloss:0.67011
[450]	validation_0-logloss:0.67004
[499]	validation_0-logloss:0.67043


In [26]:
y_preds = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

acc = metrics.accuracy_score(y_test, y_preds)
prec = metrics.precision_score(y_test, y_preds)
roc_acc = float(metrics.roc_auc_score(y_test, y_preds))

roc_acc, acc, prec

(0.5852548004040385, 0.5860045146726862, 0.5767754318618042)